# Tesi QML — Colab quickstart

Verifica il bundle PennyLane e gestisce sia la matrice primaria full sia l'analisi secondaria riprendibile sui subset 100, 300 e 900. Prima di iniziare, carica `colab_code.tar.gz` e `colab_dataset.tar.gz` nella cartella Drive `MyDrive/tesi_qml_colab`. Al primo avvio la cella di installazione riavvia automaticamente il kernel: attendi la riconnessione, poi riesegui dalla prima cella; il secondo passaggio salta `pip` ed evita il runtime NumPy misto.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/tesi_qml_colab')
PROJECT_ROOT = Path('/content/tesi_finale')
CODE_ARCHIVE = DRIVE_ROOT / 'colab_code.tar.gz'
DATASET_ARCHIVE = DRIVE_ROOT / 'colab_dataset.tar.gz'
STATE_ARCHIVE = DRIVE_ROOT / 'colab_state.tar.gz'
SUBSET_STATE_ARCHIVE = DRIVE_ROOT / 'colab_subset_state.tar.gz'

def run(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    subprocess.run(command, check=True)

for archive in (CODE_ARCHIVE, DATASET_ARCHIVE):
    if not archive.is_file():
        raise FileNotFoundError(f'Carica prima su Drive: {archive}')
print({'drive_root': str(DRIVE_ROOT), 'previous_state': STATE_ARCHIVE.is_file()})

In [ ]:
import hashlib, signal, sys

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True)
run(['tar', '-xzf', CODE_ARCHIVE, '-C', PROJECT_ROOT])
run(['tar', '-xzf', DATASET_ARCHIVE, '-C', PROJECT_ROOT])
os.chdir(PROJECT_ROOT)
if not (PROJECT_ROOT / 'requirements-colab.txt').is_file():
    shutil.copy2(PROJECT_ROOT / 'requirements-cluster.txt', PROJECT_ROOT / 'requirements-colab.txt')
requirements_path = PROJECT_ROOT / 'requirements-cluster.txt'
requirements_fingerprint = hashlib.sha256(
    sys.version.encode('utf-8') + b'\0' + requirements_path.read_bytes()
).hexdigest()
ENV_READY_MARKER = Path('/content/.tesi_qml_environment_ready')
installed_fingerprint = (
    ENV_READY_MARKER.read_text(encoding='utf-8').strip()
    if ENV_READY_MARKER.is_file()
    else None
)
if installed_fingerprint != requirements_fingerprint:
    run(['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-cluster.txt'])
    run(['python', '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'])
    ENV_READY_MARKER.write_text(requirements_fingerprint, encoding='utf-8')
    print('Dipendenze installate. Riavvio automatico del kernel: attendi la riconnessione e riesegui dalla prima cella.', flush=True)
    os.kill(os.getpid(), signal.SIGKILL)
print('Ambiente già installato: pip saltato; import eseguiti in un kernel pulito.')
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print({'device': DEVICE, 'project': str(PROJECT_ROOT)})

In [ ]:
run(['python', 'scripts/check_storage.py', '--project-root', PROJECT_ROOT, '--min-free-gb', 1, '--min-free-inodes', 1000])
run(['python', '-m', 'pytest', '-m', 'not quantum', '-q'])
run(['python', '-m', 'pytest', 'tests/quantum/test_pennylane_qnn.py', '-q'])
run(['python', 'scripts/colab_session.py', '--project-root', PROJECT_ROOT, '--state-archive', STATE_ARCHIVE, 'restore'])
run(['python', '-m', 'pytest', 'tests/test_matrix.py::test_primary_matrix_has_exactly_160_unique_iteration_one_runs', '-q'])

## Primo run

Questa cella prepara la feature bank del primo seed se manca, esegue il prossimo run pending e salva subito risultati e feature su Drive.

In [ ]:
run(['python', 'scripts/colab_session.py', '--project-root', PROJECT_ROOT, '--state-archive', STATE_ARCHIVE, 'run', '--max-runs', 1, '--device', DEVICE])
run(['python', 'scripts/colab_session.py', '--project-root', PROJECT_ROOT, '--state-archive', STATE_ARCHIVE, 'status'])

## Analisi secondaria sui subset

La matrice comprende 480 run: 2 task × 10 seed × 3 subset × 8 configurazioni. Validation e test restano identici al protocollo full; per ogni task/seed i campioni 100, 300 e 900 sono condivisi da tutti i modelli e nidificati. Lo stato incrementale viene salvato in `colab_subset_state.tar.gz` senza riscrivere a ogni blocco il voluminoso archivio full.

In [ ]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import json, subprocess, tarfile
import yaml

from tesi_qml.config import load_yaml

SUBSET_SIZES = ("100", "300", "900")
EXPECTED_RUNS_PER_SUBSET = 160
EXPECTED_SUBSET_RUNS = 480
SUBSET_SAVE_INTERVAL = 10

def subset_matrix_path(project_root, subset_size):
    return Path(project_root) / "results" / "matrix" / f"subset_{subset_size}.json"

def subset_config_path(project_root, subset_size):
    return Path(project_root) / "configs" / f"training_subset_{subset_size}.yaml"

def write_subset_training_configs(project_root, subset_sizes=SUBSET_SIZES):
    root = Path(project_root)
    base = load_yaml(root / "configs" / "training.yaml")
    paths = []
    for subset_size in subset_sizes:
        config = deepcopy(base)
        config["primary_matrix"]["subset_size"] = str(subset_size)
        path = subset_config_path(root, subset_size)
        path.write_text(
            yaml.safe_dump(config, sort_keys=False, allow_unicode=True),
            encoding="utf-8",
        )
        paths.append(path)
    return paths

def _replace_once(path, old, new, label):
    source = path.read_text(encoding="utf-8")
    if new in source:
        return False
    count = source.count(old)
    if count != 1:
        raise RuntimeError(f"Patch {label}: attesa una corrispondenza, trovate {count}")
    path.write_text(source.replace(old, new, 1), encoding="utf-8")
    return True

def prepare_subset_runtime(project_root):
    root = Path(project_root)
    changed = []
    train_path = root / "scripts" / "train.py"
    experiments_path = root / "src" / "tesi_qml" / "experiments.py"
    aggregate_path = root / "scripts" / "aggregate_results.py"
    plots_path = root / "src" / "tesi_qml" / "evaluation" / "plots.py"

    if _replace_once(
        train_path,
        '    from tesi_qml.data.features import FeatureBank\n',
        '    from tesi_qml.data.features import FeatureBank\n'
        '    from tesi_qml.reproducibility import config_fingerprint as subset_fingerprint\n',
        "import fingerprint subset",
    ):
        changed.append("train:subset-audit-import")
    if _replace_once(
        train_path,
        '    return FeatureBank(features=features, metadata=metadata, provenance=bank.provenance)\n',
        '    subset_objids = sorted(metadata["objid"].astype(str).unique())\n'
        '    provenance = {\n'
        '        **bank.provenance,\n'
        '        "training_subset_seed": int(seed),\n'
        '        "training_subset_size": len(subset_objids),\n'
        '        "training_subset_objid_sha256": subset_fingerprint(subset_objids),\n'
        '    }\n'
        '    return FeatureBank(features=features, metadata=metadata, provenance=provenance)\n',
        "audit ObjID subset",
    ):
        changed.append("train:subset-audit")
    if _replace_once(
        train_path,
        'train_bank, args.subset_size, provisional_identity["sub_seeds"]["model"]',
        'train_bank, args.subset_size, provisional_identity["sub_seeds"]["dataloader"]',
        "seed subset condiviso",
    ):
        changed.append("train:shared-subset-seed")
    if _replace_once(
        experiments_path,
        '        if self.subset_size != "full":\n'
        '            raise ValueError("Il confronto primario usa il dataset completo")\n',
        '        if self.subset_size not in {"100", "300", "900", "full"}:\n'
        '            raise ValueError("Dimensione subset non supportata")\n',
        "subset ammessi nella matrice",
    ):
        changed.append("experiments:subset-matrix")

    primary_filter = (
        '    primary_frame = frame.loc[frame["subset_size"].astype(str) == "full"].copy()\n'
    )
    aggregate_source = aggregate_path.read_text(encoding="utf-8")
    if primary_filter not in aggregate_source:
        old_validation = (
            '    if not args.allow_incomplete:\n'
            '        validate_primary_result_matrix(\n'
            '            frame, repeat_seeds=dataset_config["splits"]["repeat_seeds"]\n'
            '        )\n'
        )
        new_validation = primary_filter + old_validation.replace("            frame,", "            primary_frame,")
        if _replace_once(aggregate_path, old_validation, new_validation, "filtro matrice full"):
            changed.append("aggregate:primary-filter")
    if _replace_once(
        aggregate_path,
        '        primary = primary_hypothesis_tests(frame, seed=dataset_config["project_seed"])',
        '        primary = primary_hypothesis_tests(primary_frame, seed=dataset_config["project_seed"])',
        "ipotesi primarie solo full",
    ):
        changed.append("aggregate:primary-hypotheses")

    plots_source = plots_path.read_text(encoding="utf-8")
    if "plt.close(figure)" not in plots_source:
        lines = plots_source.splitlines()
        start = next(i for i, line in enumerate(lines) if line.startswith("def save_figure("))
        return_index = next(
            i for i in range(start, len(lines)) if lines[i].lstrip().startswith("return " )
        )
        indentation = lines[return_index][:-len(lines[return_index].lstrip())]
        lines.insert(return_index, f"{indentation}plt.close(figure)")
        plots_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        changed.append("plots:close-figure")
    return changed

def matrix_report(path, expected=EXPECTED_RUNS_PER_SUBSET):
    path = Path(path)
    if not path.is_file():
        return {"complete": 0, "failed": 0, "pending": expected, "running": 0, "total": expected}
    state = json.loads(path.read_text(encoding="utf-8"))
    counts = Counter(str(row.get("status", "pending")) for row in state.get("runs", []))
    total = int(state.get("expected_runs", expected))
    return {
        "complete": counts["complete"],
        "failed": counts["failed"],
        "pending": counts["pending"],
        "running": counts["running"],
        "total": total,
    }

def subset_reports(project_root):
    root = Path(project_root)
    return {size: matrix_report(subset_matrix_path(root, size)) for size in SUBSET_SIZES}

def _subset_state_files(project_root):
    root = Path(project_root).resolve()
    results = root / "results"
    selected = set()
    for path in (results / "matrix").glob("subset_*.json"):
        if path.is_file():
            selected.add(path.resolve())
    runs_root = results / "runs"
    if runs_root.is_dir():
        for config_path in runs_root.rglob("run_config.json"):
            try:
                config = json.loads(config_path.read_text(encoding="utf-8"))
            except (OSError, ValueError, TypeError):
                continue
            if str(config.get("subset_size")) in SUBSET_SIZES:
                selected.update(path.resolve() for path in config_path.parent.rglob("*") if path.is_file() and not path.name.endswith(".part"))
    for relative in ("aggregate", "thesis_figures"):
        directory = results / relative
        if directory.is_dir():
            selected.update(path.resolve() for path in directory.rglob("*") if path.is_file())
    log_path = results / "subset_runner.log"
    if log_path.is_file():
        selected.add(log_path.resolve())
    return sorted(selected)

def save_subset_state(project_root, archive_path):
    root = Path(project_root).resolve()
    archive = Path(archive_path).resolve()
    archive.parent.mkdir(parents=True, exist_ok=True)
    temporary = archive.with_suffix(archive.suffix + ".part")
    files = _subset_state_files(root)
    with tarfile.open(temporary, "w:gz") as bundle:
        for path in files:
            bundle.add(path, arcname=path.relative_to(root).as_posix())
    temporary.replace(archive)
    return {"archive": str(archive), "files": len(files), "size_mb": round(archive.stat().st_size / 2**20, 2)}

def restore_subset_state(project_root, archive_path):
    archive = Path(archive_path)
    if not archive.is_file():
        return False
    from scripts.colab import restore_state_archive
    return restore_state_archive(project_root, archive)

def run_subset_session(project_root, archive_path, device, max_runs, subset_sizes=SUBSET_SIZES):
    root = Path(project_root)
    if max_runs < 1:
        raise ValueError("max_runs deve essere positivo")
    subset_sizes = tuple(str(size) for size in subset_sizes)
    invalid_sizes = set(subset_sizes) - set(SUBSET_SIZES)
    if not subset_sizes or invalid_sizes:
        raise ValueError(f"Subset non validi: {sorted(invalid_sizes)}")
    reports = subset_reports(root)
    initial_complete = sum(report["complete"] for report in reports.values())
    next_save = initial_complete + SUBSET_SAVE_INTERVAL
    attempts = 0
    log_path = root / "results" / "subset_runner.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        for subset_size in subset_sizes:
            while attempts < max_runs:
                before = matrix_report(subset_matrix_path(root, subset_size))
                if before["complete"] == before["total"]:
                    break
                command = [
                    "python", str(root / "scripts" / "run_matrix.py"),
                    "--project-root", str(root),
                    "--config", str(subset_config_path(root, subset_size)),
                    "--matrix-path", str(subset_matrix_path(root, subset_size)),
                    "--max-runs", "1", "--device", str(device),
                ]
                completed = subprocess.run(command, cwd=root, text=True, capture_output=True, check=False)
                with log_path.open("a", encoding="utf-8") as log:
                    log.write("$ " + " ".join(command) + "\n")
                    log.write(completed.stdout or "")
                    log.write(completed.stderr or "")
                attempts += 1
                after = matrix_report(subset_matrix_path(root, subset_size))
                reports = subset_reports(root)
                overall_complete = sum(report["complete"] for report in reports.values())
                if completed.returncode != 0 or after["failed"]:
                    print(save_subset_state(root, archive_path), flush=True)
                    tail = log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-80:]
                    print("\n".join(tail), flush=True)
                    raise RuntimeError(f"Run subset {subset_size} fallita; stato salvato")
                if overall_complete >= next_save:
                    saved = save_subset_state(root, archive_path)
                    print({"complete": overall_complete, "total": EXPECTED_SUBSET_RUNS, "by_subset": reports, "saved": saved}, flush=True)
                    while next_save <= overall_complete:
                        next_save += SUBSET_SAVE_INTERVAL
            if attempts >= max_runs:
                break
    except KeyboardInterrupt:
        print(save_subset_state(root, archive_path), flush=True)
        raise
    saved = save_subset_state(root, archive_path)
    reports = subset_reports(root)
    print({"attempts": attempts, "by_subset": reports, "saved": saved}, flush=True)
    return reports


In [ ]:
changes = prepare_subset_runtime(PROJECT_ROOT)
subset_configs = write_subset_training_configs(PROJECT_ROOT)
subset_state_restored = restore_subset_state(PROJECT_ROOT, SUBSET_STATE_ARCHIVE)

import importlib
import scripts.train as train_module
import tesi_qml.experiments as experiments_module
import tesi_qml.matrix as matrix_module
importlib.reload(experiments_module)
importlib.reload(matrix_module)
importlib.reload(train_module)
select_training_subset = train_module.select_training_subset
from tesi_qml.data.features import FeatureBank
RunSpec = experiments_module.RunSpec
build_run_identity = experiments_module.build_run_identity
materialize_primary_matrix = matrix_module.materialize_primary_matrix
from tesi_qml.tasks import filter_feature_bank, get_task

dataset_config = load_yaml(PROJECT_ROOT / "configs" / "dataset.yaml")
full_config = load_yaml(PROJECT_ROOT / "configs" / "training.yaml")
primary_report = matrix_report(PROJECT_ROOT / "results" / "matrix" / "primary.json")
assert primary_report == {"complete": 160, "failed": 0, "pending": 0, "running": 0, "total": 160}, primary_report

full_ids = {spec.run_id for spec in materialize_primary_matrix(dataset_config, full_config)}
subset_specs = [
    spec
    for config_path in subset_configs
    for spec in materialize_primary_matrix(dataset_config, load_yaml(config_path))
]
assert len(subset_specs) == EXPECTED_SUBSET_RUNS
assert len({spec.run_id for spec in subset_specs}) == EXPECTED_SUBSET_RUNS
assert full_ids.isdisjoint({spec.run_id for spec in subset_specs})

nested_audit = []
for repeat_seed in dataset_config["splits"]["repeat_seeds"]:
    feature_root = PROJECT_ROOT / "data" / "features" / f"seed_{repeat_seed}" / "iteration_1" / "train"
    full_bank = FeatureBank.load(feature_root)
    for task_name in ("binary", "three_class"):
        task = get_task(task_name)
        task_bank = filter_feature_bank(full_bank, task)
        identity = build_run_identity(
            RunSpec(task_name, "B0", None, repeat_seed),
            dataset_config=dataset_config,
            training_config=full_config,
        )
        subset_seed = identity["sub_seeds"]["dataloader"]
        selected_sets = {
            size: set(select_training_subset(task_bank, size, subset_seed).metadata["objid"].astype(str))
            for size in SUBSET_SIZES
        }
        full_set = set(task_bank.metadata["objid"].astype(str))
        assert selected_sets["100"] < selected_sets["300"] < selected_sets["900"] < full_set
        nested_audit.append({"seed": repeat_seed, "task": task_name, "sizes": {key: len(value) for key, value in selected_sets.items()}, "full": len(full_set)})

print({
    "runtime_patches": changes,
    "subset_state_restored": subset_state_restored,
    "matrix_runs": len(subset_specs),
    "nested_audits": len(nested_audit),
    "status": subset_reports(PROJECT_ROOT),
})


### Esecuzioni separate e riprendibili

Esegui una cella per volta. Ogni subset costituisce un job indipendente da 160 run; lo stato viene mostrato e salvato su Drive ogni 10 run completate. Dopo un'interruzione, ripeti preparazione e cella del subset interessato: le run complete vengono saltate.

In [ ]:
SUBSET_TO_RUN = "100"
reports_100 = run_subset_session(
    PROJECT_ROOT,
    SUBSET_STATE_ARCHIVE,
    DEVICE,
    max_runs=160,
    subset_sizes=("100",),
)
print({"subset": SUBSET_TO_RUN, "status": reports_100[SUBSET_TO_RUN]})


In [ ]:
SUBSET_TO_RUN = "300"
reports_300 = run_subset_session(
    PROJECT_ROOT,
    SUBSET_STATE_ARCHIVE,
    DEVICE,
    max_runs=160,
    subset_sizes=("300",),
)
print({"subset": SUBSET_TO_RUN, "status": reports_300[SUBSET_TO_RUN]})


In [ ]:
SUBSET_TO_RUN = "900"
reports_900 = run_subset_session(
    PROJECT_ROOT,
    SUBSET_STATE_ARCHIVE,
    DEVICE,
    max_runs=160,
    subset_sizes=("900",),
)
print({"subset": SUBSET_TO_RUN, "status": reports_900[SUBSET_TO_RUN]})


### Aggregazione generica comparabile con il full

Puoi eseguire questa cella dopo qualsiasi subset. Aggrega tutte le run complete disponibili; i confronti Q−B2 vengono calcolati soltanto per i subset arrivati a 160/160. Genera la curva Macro-F1 disponibile e aggiorna entrambi gli archivi su Drive.

In [ ]:
reports = subset_reports(PROJECT_ROOT)
completed_subset_sizes = [
    size
    for size, report in reports.items()
    if report["complete"] == report["total"] and report["failed"] == 0
]
print({"status": reports, "subset_inclusi_nelle_statistiche": completed_subset_sizes})

run(["python", "scripts/aggregate_results.py", "--project-root", PROJECT_ROOT])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tesi_qml.cli import atomic_json, atomic_parquet
from tesi_qml.evaluation.plots import save_figure
from tesi_qml.evaluation.statistics import aggregate_runs, holm_adjust, paired_model_comparison

aggregate_dir = PROJECT_ROOT / "results" / "aggregate"
runs_frame = pd.read_parquet(aggregate_dir / "runs.parquet")
splits_frame = pd.read_parquet(PROJECT_ROOT / "data" / "splits" / "all.parquet")

full_training_counts = {}
for repeat_seed in dataset_config["splits"]["repeat_seeds"]:
    training_rows = splits_frame.loc[
        (splits_frame["repeat_seed"] == repeat_seed)
        & (splits_frame["iteration"] == 1)
        & (splits_frame["split"] == "train")
    ]
    full_training_counts[("three_class", repeat_seed)] = len(training_rows)
    full_training_counts[("binary", repeat_seed)] = int(training_rows["label"].isin(["elliptical", "spiral"]).sum())

runs_frame["training_samples"] = [
    full_training_counts[(str(row.task), int(row.repeat_seed))]
    if str(row.subset_size) == "full" else int(row.subset_size)
    for row in runs_frame.itertuples(index=False)
]
atomic_parquet(aggregate_dir / "runs_with_training_size.parquet", runs_frame)

analysis_subset_sizes = {"full", *completed_subset_sizes}
analysis_runs = runs_frame.loc[runs_frame["subset_size"].astype(str).isin(analysis_subset_sizes)].copy()
learning_summary = aggregate_runs(
    analysis_runs,
    metrics=("accuracy", "balanced_accuracy", "f1_macro", "training_seconds"),
    group_columns=("task", "model", "n_qubits", "subset_size", "training_samples"),
)
atomic_parquet(aggregate_dir / "subset_learning_curve.parquet", learning_summary)
atomic_json(aggregate_dir / "subset_learning_curve.json", learning_summary.to_dict("records"))

secondary_records = []
for subset_size in completed_subset_sizes:
    subset_index = SUBSET_SIZES.index(subset_size)
    for task_index, task_name in enumerate(("binary", "three_class")):
        for qubit_index, qubits in enumerate((4, 8)):
            selected = analysis_runs.loc[
                (analysis_runs["subset_size"].astype(str) == subset_size)
                & (analysis_runs["task"] == task_name)
                & (analysis_runs["n_qubits"] == qubits)
            ]
            comparison = paired_model_comparison(
                selected, "Q", "B2", "f1_macro",
                seed=dataset_config["project_seed"] + subset_index * 4 + task_index * 2 + qubit_index,
            )
            secondary_records.append({"subset_size": subset_size, "task": task_name, "n_qubits": qubits, **comparison, "raw_p_value": comparison["permutation_p_value"]})
if secondary_records:
    secondary_adjusted = holm_adjust([record["raw_p_value"] for record in secondary_records])
    for record, adjusted in zip(secondary_records, secondary_adjusted):
        record["holm_p_value"] = float(adjusted)
secondary_frame = pd.DataFrame(secondary_records)
if not secondary_frame.empty:
    atomic_parquet(aggregate_dir / "secondary_subset_hypotheses.parquet", secondary_frame)
atomic_json(aggregate_dir / "secondary_subset_hypotheses.json", secondary_frame.to_dict("records"))

plot_frame = learning_summary.copy()
plot_frame["model_label"] = [
    str(row.model) if pd.isna(row.n_qubits) else f"{row.model}-{int(row.n_qubits)}"
    for row in plot_frame.itertuples(index=False)
]
figure, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for axis, task_name in zip(axes, ("binary", "three_class")):
    task_rows = plot_frame.loc[plot_frame["task"] == task_name]
    for label, group in task_rows.groupby("model_label", sort=True):
        group = group.sort_values("training_samples")
        means = group["f1_macro_mean"].to_numpy(float)
        lower = means - group["f1_macro_ci95_low"].to_numpy(float)
        upper = group["f1_macro_ci95_high"].to_numpy(float) - means
        axis.errorbar(group["training_samples"], means, yerr=np.vstack([lower, upper]), marker="o", capsize=3, label=label)
    axis.set(xlabel="Campioni di training", ylabel="Macro-F1", title=task_name, ylim=(0, 1))
    axis.legend(fontsize="small", ncol=2)
save_figure(figure, PROJECT_ROOT / "results" / "thesis_figures" / "macro_f1_vs_training_samples")

incremental = save_subset_state(PROJECT_ROOT, SUBSET_STATE_ARCHIVE)
run(["python", "scripts/colab_session.py", "--project-root", PROJECT_ROOT, "--state-archive", STATE_ARCHIVE, "save"])
print({"full_and_subsets_saved": str(STATE_ARCHIVE), "incremental_backup": incremental, "runs": len(runs_frame), "secondary_tests": len(secondary_frame)})
